# MineColab Improved — Modern Paper Edition

Run a persistent Minecraft Java server from Google Drive with the latest supported Paper build and a Playit.gg public address.

> **Important:** Colab is not intended for permanent game hosting. Sessions can stop without warning, and Google may restrict this workload. Keep backups and follow the platform's terms.

Run the cells from top to bottom. Re-running the install cell safely checks for a newer Paper build.

## 1. Configuration

`latest` means the newest Minecraft version that has a build in the selected Paper channel. `STABLE` is recommended.

In [ ]:
#@title Server configuration
MINECRAFT_VERSION = "latest" #@param {type:"string"}
PAPER_CHANNEL = "STABLE" #@param ["STABLE", "BETA", "ALPHA"]
RAM_GB = 6 #@param {type:"integer"}
SERVER_PORT = 25565 #@param {type:"integer"}
MOTD = "MineColab Paper Server" #@param {type:"string"}
MAX_PLAYERS = 20 #@param {type:"integer"}
ONLINE_MODE = True #@param {type:"boolean"}
VIEW_DISTANCE = 8 #@param {type:"integer"}
SIMULATION_DISTANCE = 6 #@param {type:"integer"}
AUTO_UPDATE_PAPER = True #@param {type:"boolean"}
TUNNEL_SERVICE = "playit" #@param ["playit", "none"]

SERVER_DIR = "/content/drive/MyDrive/Minecraft-server"
JAVA_VERSION = 25
PAPER_USER_AGENT = "MineColab-Improved/2.0 (https://github.com/N-aksif-N/MineColab_Improved)"

if RAM_GB < 2:
    raise ValueError("RAM_GB must be at least 2")
if not 1 <= SERVER_PORT <= 65535:
    raise ValueError("SERVER_PORT must be between 1 and 65535")
if PAPER_CHANNEL not in {"STABLE", "BETA", "ALPHA"}:
    raise ValueError("Unsupported Paper channel")
if TUNNEL_SERVICE not in {"playit", "none"}:
    raise ValueError("Unsupported tunnel service")
print("Configuration accepted.")

## 2. Mount Drive and install Java 25

Java is installed into the temporary Colab runtime. Your Minecraft files remain in Google Drive.

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import tarfile
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import requests
from google.colab import drive

drive.mount("/content/drive")
server_path = Path(SERVER_DIR)
server_path.mkdir(parents=True, exist_ok=True)
for folder in ("plugins", "backups", "logs"):
    (server_path / folder).mkdir(exist_ok=True)

def download_file(url, destination, headers=None, expected_sha256=None):
    destination = Path(destination)
    digest = hashlib.sha256()
    with requests.get(url, headers=headers, stream=True, timeout=(20, 300)) as response:
        response.raise_for_status()
        with destination.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
                    digest.update(chunk)
    actual = digest.hexdigest()
    if expected_sha256 and actual.lower() != expected_sha256.lower():
        destination.unlink(missing_ok=True)
        raise RuntimeError(f"Checksum mismatch for {destination.name}")
    return actual

machine = platform.machine().lower()
architecture = {"x86_64": "x64", "amd64": "x64", "aarch64": "aarch64", "arm64": "aarch64"}.get(machine)
if architecture is None:
    raise RuntimeError(f"Unsupported CPU architecture: {machine}")

java_home = Path(f"/content/temurin-{JAVA_VERSION}")
JAVA_BIN = java_home / "bin/java"
if not JAVA_BIN.exists():
    print(f"Installing Eclipse Temurin Java {JAVA_VERSION}...")
    api = (f"https://api.adoptium.net/v3/assets/latest/{JAVA_VERSION}/hotspot"
           f"?architecture={architecture}&image_type=jdk&os=linux&vendor=eclipse")
    assets = requests.get(api, timeout=30).json()
    if not assets:
        raise RuntimeError("Adoptium returned no compatible Java build")
    package = assets[0]["binary"]["package"]
    archive = Path(f"/content/temurin-{JAVA_VERSION}.tar.gz")
    download_file(package["link"], archive, expected_sha256=package["checksum"])
    shutil.rmtree(java_home, ignore_errors=True)
    java_home.mkdir(parents=True)
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(java_home), "--strip-components=1"], check=True)
    archive.unlink(missing_ok=True)

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = f"{java_home / 'bin'}:{os.environ['PATH']}"
subprocess.run([str(JAVA_BIN), "-version"], check=True)
print(f"Server directory: {server_path}")

## 3. Install or update Paper

This selects the newest matching Paper build, verifies its SHA-256 checksum, and keeps the previous JAR in `backups/` when upgrading.

In [ ]:
PAPER_API = "https://fill.papermc.io/v3"
paper_headers = {"User-Agent": PAPER_USER_AGENT, "Accept": "application/json"}

def get_json(url):
    response = requests.get(url, headers=paper_headers, timeout=30)
    response.raise_for_status()
    return response.json()

project = get_json(f"{PAPER_API}/projects/paper")
available_versions = [version for group in project["versions"].values() for version in group]
candidate_versions = available_versions if MINECRAFT_VERSION.lower() == "latest" else [MINECRAFT_VERSION]
selected_version = None
selected_build = None

for candidate in candidate_versions:
    if candidate not in available_versions:
        continue
    builds = get_json(f"{PAPER_API}/projects/paper/versions/{candidate}/builds")
    matching = [build for build in builds if build["channel"].upper() == PAPER_CHANNEL]
    if matching:
        selected_version = candidate
        selected_build = max(matching, key=lambda build: build["id"])
        break

if selected_build is None:
    raise RuntimeError(f"No {PAPER_CHANNEL} Paper build found for {MINECRAFT_VERSION}")

download = selected_build["downloads"]["server:default"]
expected_sha = download["checksums"]["sha256"]
server_jar = server_path / "server.jar"
config_file = server_path / "minecolab-config.json"
old_config = json.loads(config_file.read_text()) if config_file.exists() else {}
already_current = server_jar.exists() and old_config.get("paper_sha256") == expected_sha

if already_current:
    print(f"Paper {selected_version} build {selected_build['id']} is already installed.")
elif server_jar.exists() and not AUTO_UPDATE_PAPER:
    print("A Paper update is available, but AUTO_UPDATE_PAPER is disabled.")
else:
    temporary_jar = server_path / "server.jar.download"
    print(f"Downloading Paper {selected_version} build {selected_build['id']} ({PAPER_CHANNEL})...")
    download_file(download["url"], temporary_jar, headers=paper_headers, expected_sha256=expected_sha)
    if server_jar.exists():
        old_version = old_config.get("minecraft_version", "unknown")
        old_build = old_config.get("paper_build", "unknown")
        backup_jar = server_path / "backups" / f"paper-{old_version}-{old_build}.jar"
        if not backup_jar.exists():
            shutil.copy2(server_jar, backup_jar)
    temporary_jar.replace(server_jar)
    config = {
        "minecraft_version": selected_version,
        "paper_build": selected_build["id"],
        "paper_channel": PAPER_CHANNEL,
        "paper_sha256": expected_sha,
        "updated_at": datetime.now(timezone.utc).isoformat(),
    }
    config_file.write_text(json.dumps(config, indent=2) + "\n")
    print("Paper installed and checksum verified.")

## 4. Accept the EULA and write server settings

Read the [Minecraft EULA](https://www.minecraft.net/eula), then change the option below to `True`.

In [ ]:
ACCEPT_MINECRAFT_EULA = False #@param {type:"boolean"}

if not ACCEPT_MINECRAFT_EULA:
    raise RuntimeError("Read https://www.minecraft.net/eula, set ACCEPT_MINECRAFT_EULA=True, and run this cell again.")

(server_path / "eula.txt").write_text("eula=true\n")

properties_path = server_path / "server.properties"
existing_lines = properties_path.read_text(errors="replace").splitlines() if properties_path.exists() else []
settings = {
    "server-port": str(SERVER_PORT),
    "motd": MOTD.replace("\n", " ").replace("\r", " "),
    "max-players": str(MAX_PLAYERS),
    "online-mode": str(ONLINE_MODE).lower(),
    "view-distance": str(VIEW_DISTANCE),
    "simulation-distance": str(SIMULATION_DISTANCE),
}
seen = set()
updated_lines = []
for line in existing_lines:
    key = line.split("=", 1)[0] if "=" in line and not line.lstrip().startswith("#") else None
    if key in settings:
        updated_lines.append(f"{key}={settings[key]}")
        seen.add(key)
    else:
        updated_lines.append(line)
for key, value in settings.items():
    if key not in seen:
        updated_lines.append(f"{key}={value}")
properties_path.write_text("\n".join(updated_lines) + "\n")
print("EULA accepted and server.properties updated.")

## 5. Install and claim Playit.gg

Run once per new Colab runtime. Open the claim URL printed by Playit, sign in, create a **Minecraft Java** tunnel, and point it to `127.0.0.1:25565` (or your configured port). Your public address is shown in the Playit dashboard and agent output.

In [ ]:
if TUNNEL_SERVICE == "playit":
    if shutil.which("playit") is None:
        print("Installing Playit from the official package repository...")
        subprocess.run(
            ["bash", "-lc", "curl -fsSL https://packages.playit.gg/install.sh | bash -s -- -y"],
            check=True,
        )
    subprocess.run(["playit", "setup"], check=True)
else:
    print("Tunnel disabled; players must connect through another network solution.")

## 6. Start the tunnel and Minecraft server

Leave this cell running. Use the Minecraft Java address assigned in your Playit dashboard. Stop the cell to shut down the server and tunnel cleanly.

In [ ]:
import signal
import time

if not (server_path / "server.jar").exists():
    raise FileNotFoundError("server.jar is missing; run the Paper install cell first")
if not (server_path / "eula.txt").exists():
    raise FileNotFoundError("eula.txt is missing; accept the EULA first")

playit_process = None
server_process = None
try:
    if TUNNEL_SERVICE == "playit":
        playit_process = subprocess.Popen(["playit"], cwd=server_path)
        time.sleep(4)
        subprocess.run(["playit", "tunnels", "list"], check=False)

    java_command = [
        str(JAVA_BIN),
        f"-Xms{RAM_GB}G",
        f"-Xmx{RAM_GB}G",
        "-XX:+UseG1GC",
        "-XX:+ParallelRefProcEnabled",
        "-XX:MaxGCPauseMillis=200",
        "-XX:+DisableExplicitGC",
        "-jar",
        "server.jar",
        "--nogui",
    ]
    print("Starting Paper. Type commands in the server console below.")
    server_process = subprocess.Popen(java_command, cwd=server_path)
    server_process.wait()
except KeyboardInterrupt:
    print("Stopping server...")
    if server_process and server_process.poll() is None:
        server_process.send_signal(signal.SIGINT)
        try:
            server_process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            server_process.terminate()
finally:
    if playit_process and playit_process.poll() is None:
        playit_process.terminate()
        try:
            playit_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            playit_process.kill()
    print("Server and tunnel stopped.")

## 7. Create a backup

Stop the server before backing up so the world files are consistent. Backups are stored in Google Drive.

In [ ]:
if server_process is not None and server_process.poll() is None:
    raise RuntimeError("Stop the server before creating a backup")
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
backup_base = server_path / "backups" / f"minecraft-server-{timestamp}"
archive_path = shutil.make_archive(
    str(backup_base),
    "zip",
    root_dir=server_path,
    base_dir=".",
)
print(f"Backup created: {archive_path}")

## Plugins and administration

Upload Paper-compatible plugin JAR files to `Minecraft-server/plugins`, then restart the server. Only download plugins from sources you trust, and confirm that each plugin supports the installed Minecraft/Paper version. Server logs are stored in `Minecraft-server/logs`.